## Evaluate integration models

In [ ]:
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

## Load Libraries

In [ ]:
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [ ]:
# Paths 
adata_dir = str(here('data/anndata/'))
# Saving
save_dir = str(here('data/integration/first_pass/'))
plot_dir = str(here('data/integration/first_pass/plot/'))
tmp_dir = str(here('data/integration/first_pass/tmp/'))

In [ ]:
base_dir = str(here('data/integration/first_pass/'))
hvgs = [500, 1000, 2000]
methods = ["unintegrated", "scvi", "sysvi", "scpoli"]
adata_dir = "adata_hvg_subsets"  # directory with preprocessed HVG files
file_dir = os.path.join(base_dir, 'files')  # directory to save embeddings
model_dir = os.path.join(base_dir, 'models')   # directory to save models
key_batch = ["condition", "batch"]   # adjust to your obs keys
key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"
latent_dims = 20
embedding_dims = 20
label_key = 'study_cell_annotation_harmonized' 

## Load data

In [ ]:
adata = sc.read_h5ad("adata_full_reference.h5ad")

In [ ]:
# subset for test
# number per group
n_per_group = 3000 // adata.obs['name'].nunique()

# sample indices
sampled_obs = (
    adata.obs
    .groupby('name', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), n_per_group), random_state=0))
)

# subset AnnData
adata_subset = adata[sampled_obs.index, :].copy()

In [ ]:
adata.raw = adata.copy()

# make na unknown, otherwise it will not work
adata.obs[label_key] = adata.obs[label_key].fillna('unknown')

In [ ]:
for n in hvgs:
    
    ad = adata.copy()
    
    sc.pp.highly_variable_genes(
            a,
            n_top_genes=n,
            flavor="seurat_v3",
            layer="counts",
            batch_key=batch_key,
            subset=True
        )

    #!!!!!!! possibly added traning plots to code
    
    # Run unintegrated (PCA)
    subprocess.run([
        "conda", "run", "-n", "py_analysis", "python", "run_no_int.py",
        n, adata_file, model_dir, file_dir, key_save, latent_dims
    ])
    

    # Run scVI
    subprocess.run([
        "conda", "run", "-n", "py_analysis, "python", "run_scvi.py",
        n, adata_file, model_dir, file_dir, key_batch[0], key_batch[1], key_save, latent_dims, embedding_dims
    ])
    
    # Run SysVI
    subprocess.run([
        "conda", "run", "-n", "py_analysis", "python", "run_sysvi.py",
        n, adata_file, model_dir, file_dir, key_batch[0], key_batch[1], key_save, latent_dims, embedding_dims
    ])
    
    # Run scPoli
    subprocess.run([
        "conda", "run", "-n", "scpoli", "python", "run_scpoli.py",
        n, adata_file, model_dir, file_dir, key_batch[1], key_save, latent_dims, embedding_dims
    ])

In [ ]:
latent_keys = [key for key in adata.obsm.keys() if any(m in key for m in methods)]

for key in latent_keys:
    print(f"Computing neighbors and UMAP for {key}...")
    
    # Compute neighbors on the latent space
    sc.pp.neighbors(adata, use_rep=key, key_added=f"{key}_neighbors")
    
    # Compute UMAP, store in obsm
    sc.tl.umap(adata, neighbors_key=f"{key}_neighbors", key_added=f"{key}_umap")
    
    # Optional: plot
    sc.pl.umap(adata, color="study_cell_annotation_harmonized", 
               use_rep=f"{key}_umap", 
               title=f"UMAP {key}", 
               save=os.path.join("figures", f"UMAP_{key}.png"))

In [ ]:
for method in methods:
    for n in hvgs:
        df_path = os.path.join(file_dir, f"{method}_latent_embd_{n}_{key_save}.csv")
        if not os.path.exists(df_path):
            print(f' Missing embedding: {df_path}')
            continue
        
        latent_df = pd.read_csv(df_path, index_col=0)
        latent_df = latent_df.loc[adata.obs_names]  # ensure correct cell order
        adata.obsm[f"{method}_{n}"] = latent_df.to_numpy()

